## Transformation logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_key) AS product_key,
    pn.product_id       AS product_id,
    pn.product_key      AS product_number,
    pn.product_name       AS product_name,
    pn.category_id       AS category_id,
    pc.category          AS category,
    pc.subcategory       AS subcategory,
    pc.maintenance_flag  AS maintenance_flag,
    pn.product_cost     AS cost,
    pn.product_line     AS product_line,
    pn.start_date AS start_date
FROM silver.crm_products pn
LEFT JOIN silver.erp_product_category pc
    ON pn.category_id = pc.category_id
WHERE pn.end_date IS NULL; --Filter out all historical date
"""

df = spark.sql(query)

## Sanity check of dataframe

In [0]:
df.limit(10).display()


product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,cost,product_line,start_date
1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
6,351,BK-M82B-48,Mountain-100 Black- 48,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
7,344,BK-M82S-38,Mountain-100 Silver- 38,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
8,345,BK-M82S-42,Mountain-100 Silver- 42,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
9,346,BK-M82S-44,Mountain-100 Silver- 44,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
10,347,BK-M82S-48,Mountain-100 Silver- 48,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01


In [0]:
%sql
SHOW TABLES FROM workspace.silver


database,tableName,isTemporary
silver,crm_customers,false
silver,crm_products,false
silver,crm_sales,false
silver,erp_customer_location,false
silver,erp_customers,false
silver,erp_product_category,false


## Write gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")

## Sanity check of gold table

In [0]:
%sql
SELECT *
FROM workspace.gold.dim_products
LIMIT 10

product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,cost,product_line,start_date
1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
6,351,BK-M82B-48,Mountain-100 Black- 48,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
7,344,BK-M82S-38,Mountain-100 Silver- 38,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
8,345,BK-M82S-42,Mountain-100 Silver- 42,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
9,346,BK-M82S-44,Mountain-100 Silver- 44,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
10,347,BK-M82S-48,Mountain-100 Silver- 48,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
